## Example 1: Check a clean wheel

In [ ]:
import importlib.metadata
import tempfile
import zipfile
from pathlib import Path

from check_wheel_contents.checker import WheelChecker
from check_wheel_contents.contents import WheelContents

# Example 1: a tiny well-formed wheel built by hand passes every default check
assert importlib.metadata.version("check-wheel-contents") == "0.6.3"
tmp = Path(tempfile.mkdtemp())
whl = tmp / "demo-1.0-py3-none-any.whl"
with zipfile.ZipFile(whl, "w") as zf:
    zf.writestr("demo/__init__.py", "VALUE = 1\n")
    zf.writestr("demo-1.0.dist-info/METADATA", "Metadata-Version: 2.1\nName: demo\nVersion: 1.0\n")
    zf.writestr("demo-1.0.dist-info/WHEEL", "Wheel-Version: 1.0\nRoot-Is-Purelib: true\nTag: py3-none-any\n")
    zf.writestr("demo-1.0.dist-info/RECORD", "".join(f"{n},,\n" for n in zf.namelist()) + "demo-1.0.dist-info/RECORD,,\n")
failures = WheelChecker().check_contents(WheelContents.from_wheel(whl))
assert failures == [], [f.show() for f in failures]
print("Example 1 OK: no failures")

## Example 2: Stray bytecode and a top-level tests package

In [ ]:
import tempfile
import zipfile
from pathlib import Path

from check_wheel_contents.checker import WheelChecker
from check_wheel_contents.contents import WheelContents

# Example 2: W001 (.pyc in the wheel) and W005 (a common top-level name such as tests)
whl = Path(tempfile.mkdtemp()) / "demo-1.0-py3-none-any.whl"
with zipfile.ZipFile(whl, "w") as zf:
    zf.writestr("demo/__init__.py", "")
    zf.writestr("demo/__pycache__/__init__.cpython-314.pyc", b"\0")
    zf.writestr("tests/__init__.py", "")
    zf.writestr("demo-1.0.dist-info/METADATA", "Metadata-Version: 2.1\nName: demo\nVersion: 1.0\n")
    zf.writestr("demo-1.0.dist-info/WHEEL", "Wheel-Version: 1.0\nRoot-Is-Purelib: true\nTag: py3-none-any\n")
    zf.writestr("demo-1.0.dist-info/RECORD", "".join(f"{n},,\n" for n in zf.namelist()) + "demo-1.0.dist-info/RECORD,,\n")
names = sorted(f.check.name for f in WheelChecker().check_contents(WheelContents.from_wheel(whl)))
assert "W001" in names and "W005" in names, names
print("Example 2 OK:", names)

## Example 3: Select checks by prefix

In [ ]:
from check_wheel_contents.checks import Check, parse_checks_string

# Example 3: --select / --ignore take check names or prefixes ("W00" = W001-W009, "W0" adds W010)
assert parse_checks_string("W001,W1") == {Check.W001, Check.W101, Check.W102}
assert len(parse_checks_string("W00")) == 9 and len(parse_checks_string("W0")) == 10
assert Check.W002.value == "Wheel contains duplicate files"
print("Example 3 OK:", len(Check), "checks")

## Example 4: The command line and its exit code

In [ ]:
import shutil
import subprocess
import tempfile
import zipfile
from pathlib import Path

# Example 4: check-wheel-contents exits 1 and names the check on a bad wheel, 0 on a good one
exe = shutil.which("check-wheel-contents")
assert exe, "check-wheel-contents is not on PATH"
whl = Path(tempfile.mkdtemp()) / "solo-1.0-py3-none-any.whl"
with zipfile.ZipFile(whl, "w") as zf:
    zf.writestr("solo/__init__.py", "x = 1\n")
    zf.writestr("solo/__init__.pyc", b"\0")
    zf.writestr("solo-1.0.dist-info/WHEEL", "Wheel-Version: 1.0\nRoot-Is-Purelib: true\nTag: py3-none-any\n")
    zf.writestr("solo-1.0.dist-info/RECORD", "".join(f"{n},,\n" for n in zf.namelist()) + "solo-1.0.dist-info/RECORD,,\n")
bad = subprocess.run([exe, str(whl)], capture_output=True, text=True)
assert bad.returncode == 1 and "W001" in bad.stdout, (bad.returncode, bad.stdout, bad.stderr)
good = subprocess.run([exe, "--ignore", "W001", str(whl)], capture_output=True, text=True)
assert good.returncode == 0, (good.stdout, good.stderr)
print("Example 4 OK:", bad.stdout.strip().splitlines()[0])

## Example 5: Check a wheel that pip built

In [ ]:
import pathlib
import subprocess
import sys
import tempfile

from check_wheel_contents.checker import WheelChecker
from check_wheel_contents.contents import WheelContents

# Example 5: build a real wheel from a small project with pip (no network, no build isolation) and check it
src = pathlib.Path(tempfile.mkdtemp())
(src / "hello").mkdir()
(src / "hello" / "__init__.py").write_text("def hi():\n    return 'hi'\n")
(src / "pyproject.toml").write_text('[build-system]\nrequires = ["setuptools"]\nbuild-backend = "setuptools.build_meta"\n'
                                    '[project]\nname = "hello"\nversion = "0.1"\n')
out = src / "dist"
subprocess.run([sys.executable, "-m", "pip", "wheel", "--no-deps", "--no-build-isolation", "--no-index", "-q", "-w", str(out), str(src)], check=True)
whl = next(out.glob("hello-0.1-*.whl"))
failures = WheelChecker().check_contents(WheelContents.from_wheel(whl))
assert failures == [], [f.show() for f in failures]
print("Example 5 OK:", whl.name)